# Notebook 05 — Train ResNet-50 Disease Detector → Export to ONNX

Fine-tunes a ResNet-50 on PlantVillage (or custom) disease data and exports to ONNX.

In [ ]:
import os, json, hashlib, time
from pathlib import Path

DATA_DIR = Path(os.getenv('DISEASE_DATA_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\data\diseases'))
EXPORTS_DIR = Path(os.getenv('EXPORTS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\exports\models'))
RUNS_DIR = Path(os.getenv('RUNS_DIR', r'C:\Users\MJ\Desktop\Agric\jupyter\runs'))
MODEL_VERSION = os.getenv('MODEL_VERSION', 'v1')
IMG_SIZE = int(os.getenv('DISEASE_IMG_SIZE', '224'))
BATCH = int(os.getenv('DISEASE_BATCH', '32'))
EPOCHS = int(os.getenv('DISEASE_EPOCHS', '20'))
LR = float(os.getenv('DISEASE_LR', '1e-4'))
MIN_IMAGES_PER_CLASS = int(os.getenv('MIN_DISEASE_IMAGES_PER_CLASS', '10'))
MIN_VAL_ACCURACY = float(os.getenv('MIN_DISEASE_VAL_ACCURACY', '0.70'))

EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

DISEASE_CLASSES = [
    'healthy', 'rust', 'blight', 'powdery_mildew',
    'mosaic_virus', 'leaf_spot', 'rot',
]
NUM_CLASSES = len(DISEASE_CLASSES)

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('Disease detector configuration')
print('DATA_DIR:', DATA_DIR)
print('EXPORTS_DIR:', EXPORTS_DIR)
print('MODEL_VERSION:', MODEL_VERSION)
print(f'Classes ({NUM_CLASSES}): {DISEASE_CLASSES}')

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import datasets, models
from torch.utils.data import DataLoader

train_dir = DATA_DIR / 'train'
val_dir = DATA_DIR / 'val'
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Disease dataset not found: {DATA_DIR}. Place your dataset under train/ and val/ class folders.')
if not train_dir.exists():
    raise FileNotFoundError(f'Missing disease training folder: {train_dir}')
if not val_dir.exists():
    raise FileNotFoundError(f'Missing disease validation folder: {val_dir}')

class_counts = {}
for cls in DISEASE_CLASSES:
    cls_dir = train_dir / cls
    images = []
    if cls_dir.exists():
        for pattern in ('*.jpg', '*.jpeg', '*.png', '*.webp'):
            images.extend(cls_dir.glob(pattern))
    class_counts[cls] = len(images)

missing_or_small = {cls: n for cls, n in class_counts.items() if n < MIN_IMAGES_PER_CLASS}
print('Training image counts:', class_counts)
if missing_or_small:
    raise ValueError(f'Insufficient disease images per class. Need >= {MIN_IMAGES_PER_CLASS}: {missing_or_small}')

transform_train = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_val = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(train_dir, transform=transform_train)
val_ds = datasets.ImageFolder(val_dir, transform=transform_val)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print('Detected class mapping:', train_ds.class_to_idx)

In [ ]:
device = torch.device(os.getenv('TRAIN_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)

backbone = models.resnet50(weights='IMAGENET1K_V2')
backbone.fc = nn.Linear(backbone.fc.in_features, NUM_CLASSES)
model = backbone.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

best_acc = 0.0
best_path = RUNS_DIR / 'disease_best.pth'
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dl:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for inputs, labels in val_dl:
            inputs, labels = inputs.to(device), labels.to(device)
            preds = model(inputs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / max(total, 1)
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_path)
    scheduler.step()
    print(f'Epoch {epoch+1}/{EPOCHS}  loss={running_loss/max(len(train_dl), 1):.4f}  val_acc={acc:.2%}')

print(f'Best validation accuracy: {best_acc:.2%}')
assert best_acc >= MIN_VAL_ACCURACY, f'Accuracy {best_acc:.2%} below {MIN_VAL_ACCURACY:.0%} — collect more data'

In [ ]:
model.load_state_dict(torch.load(best_path, map_location='cpu'))
model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = EXPORTS_DIR / 'disease_detector_v1.onnx'

torch.onnx.export(
    model, dummy, str(onnx_path),
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=17,
)
print('Exported:', onnx_path)

In [ ]:
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy_np})
print('ONNX smoke-test passed. Output shape:', out[0].shape)

meta = {
    'model': 'disease_detector_v1',
    'version': MODEL_VERSION,
    'format': 'onnx',
    'source_notebook': '05_train_disease_detector.ipynb',
    'dataset_dir': str(DATA_DIR),
    'dataset_class_counts': class_counts,
    'input_size': IMG_SIZE,
    'classes': DISEASE_CLASSES,
    'detected_class_to_idx': train_ds.class_to_idx,
    'val_accuracy': round(float(best_acc), 4),
    'minimum_val_accuracy': MIN_VAL_ACCURACY,
    'sha256': sha256_file(onnx_path),
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
metadata_path = EXPORTS_DIR / 'disease_detector_metadata.json'
metadata_path.write_text(json.dumps(meta, indent=2))
print('Metadata saved:', metadata_path)
print(json.dumps(meta, indent=2))